# Ecossistema Google para desenvolvimento de Agents
*Sauter Digital - 2026*

**authors**

- *Gabriel Carvalho*

Documentation https://adk.dev/get-started/

![Static Badge](https://img.shields.io/badge/python-3.12.12-blue?logo=python)
![Static Badge](https://img.shields.io/badge/Sauter%20Digital-2026-7201E8)
[![GitHub](https://img.shields.io/badge/github-code-black?logo=github)](https://github.com/iamgabs/upe-sauter-ecossistema-google)

## Setup

In [24]:
# @title Setup: instalação do ADK

!pip install google-adk -q

In [25]:
# @title ignorando warnings e definindo o nível do log

import warnings

warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.ERROR)


In [26]:
# @title Runner (específico para o Google Colab)
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

def adk_runner(agent, app_name, session_service=None):
    if session_service is None:
        session_service = InMemorySessionService()

    return Runner(
        agent=agent,
        app_name=app_name,
        session_service=session_service,
    )


async def call_agent_async(query: str, runner, user_id, session_id):
    content = types.Content(role="user", parts=[types.Part(text=query)])

    # default
    final_response_text = "O Agente não conseguiu produzir uma resposta"

    async for event in runner.run_async(
        user_id=user_id, session_id=session_id, new_message=content
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            elif (
                event.actions and event.actions.escalate
            ):  # Handle potential errors/escalations
                final_response_text = (
                    f"Agent escalated: {event.error_message or 'No specific message.'}"
                )
            break

    print(f"<<<< Reposta do Agente: {final_response_text}")

### Configuração das credenciais e inicialização do modelo

Para interagir com os modelos do Google (como o Gemma e o Gemini), precisaremos configurar a nossa chave de API. Certifique-se de adicionar a sua chave nos secrets do colab (ícone de chave na barra lateral esquerda) com o nome `GEMINI_API_KEY` ou `GOOGLE_API_KEY`.

Para criar uma chave de API, devemos ir até o Ai Studio (ai.dev), no menu lateral, no canto inferior esquerdo, clique no ícone de chave, e posteriormente em `Create API Key`, a sua chave é única, e possui um sistema de TIERS, inicialmente com o `FREE TIER`, que permite você realizar um número de requests máximo por dia.


In [ ]:
from google.colab import userdata
import os

# Recupera a chave de API dos segredos do Colab
try:
    api_key = userdata.get('<SUA_API_KEY>')
    os.environ['GOOGLE_API_KEY'] = api_key
    print("Chave de API configurada com sucesso!")
except Exception as e:
    print("Por favor, adicione a sua chave 'GOOGLE_API_KEY' nos Secrets do Colab.")

Chave de API configurada com sucesso!


## Criando o Agente com o ADK

Agora, vamos fazer o nosso agente se comunicar com uma tool externa. Uma tool pode ser uma função simples, na prática, não é tão simples para utilizar com coisas mais complexas, porém majoritariamente, será uma definição de função simples

In [29]:
# @title nossa primeira tool

def consultar_cotacao_moeda(moeda: str) -> str:
  return "BRL to USD (0.20), BRL to EUR (0.18)"

In [34]:
# @title tool real (ana maria braga)
def consultar_endereco_por_cep(cep: str) -> str:
    import urllib.request
    import json

    cep_limpo = "".join(filter(str.isdigit, cep))

    if len(cep_limpo) != 8:
        return "Erro: O CEP deve conter exatamente 8 dígitos numéricos."

    url = f"https://viacep.com.br/ws/{cep_limpo}/json/"

    try:
        with urllib.request.urlopen(url) as response:
            dados = json.loads(response.read().decode('utf-8'))

            if "erro" in dados and dados["erro"]:
                return f"CEP {cep_limpo} não encontrado na base de dados."

            logradouro = dados.get("logradouro", "Não informado")
            bairro = dados.get("bairro", "Não informado")
            cidade = dados.get("localidade", "Não informada")
            uf = dados.get("uf", "")

            return f"Endereço: {logradouro}, {bairro} - {cidade}/{uf}."

    except Exception as e:
        return f"Falha ao conectar ao serviço de CEP: {str(e)}"

In [35]:
from google.adk.agents import LlmAgent

# Criando um agente simples com os parâmetros validados pelo ADK
root_agent = LlmAgent(
    name="assistente_amigavel",
    instruction=("Você é um assistente útil, amigável e conciso que fala em português."
      "Quando o usuário fornecer ou perguntar sobre um CEP ou pedir para localizar um endereço, "
      "use a ferramenta consultar_endereco_por_cep."),
    model="gemini-flash-latest",
    tools=[consultar_endereco_por_cep]
)

In [9]:
# @title definindo os parâmetros de sessão
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()

APP_NAME = "upe_first_agent"
USER_ID = "user_1"
SESSION_ID = "session_001"

session = await session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
)

In [36]:
# @title Interação com o Agente

runner = adk_runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

user_input = "Qual é a rua de cep 55293-000" # @param {type:"string"}

await call_agent_async(
    query=user_input,
    runner=runner,
    user_id=USER_ID,
    session_id=SESSION_ID
)

<<<< Reposta do Agente: O CEP **55293-000** pertence à **Avenida Santo Antônio**, no bairro Santo Antônio, em Garanhuns/PE.
